In [ ]:
# %%
# SECTION 1: Imports
# -------------------
# Loads all required libraries:
#   - playwright sync_playwright: controls a real Chromium browser for JS-rendered pages.
#     Unlike Selenium, Playwright bundles its own browser — no ChromeDriver needed.
#   - BeautifulSoup: parses the rendered HTML to extract data
#   - csv: writes extracted records to a CSV file
#
# Requirements:
#   pip install playwright beautifulsoup4
#   playwright install chromium

import csv

from bs4 import BeautifulSoup
from playwright.sync_api import TimeoutError as PlaywrightTimeoutError
from playwright.sync_api import sync_playwright

print("Imports loaded successfully.")

In [ ]:
# %%
# SECTION 2: Configuration
# -------------------------
# Set your scrape target and options here before running any section below.
#
# url          - The page you want to scrape
# wait_for     - CSS selector that only appears once JS has rendered the data.
#                Playwright waits until this selector is visible before capturing HTML.
# timeout      - Max milliseconds to wait (Playwright uses ms, not seconds)
# headless     - True: no browser window. False: shows browser (good for debugging).
# parent_tag   - HTML tag wrapping each record
# parent_class - CSS class to filter parent elements (None to match all)
# child_tag    - Tag to search within each parent element
# fields       - List of (column_name, attr_name, attr_value) tuples
# output_path  - Where to save the resulting CSV

url          = 'https://www.cdms.net/Label-Database/Advanced-Search#Result-products'
wait_for     = "span[databind='text: name']"  # CSS selector to confirm JS has loaded
timeout      = 15000  # milliseconds (15s) — increase for slow pages
headless     = True   # Set to False to watch the browser while debugging

parent_tag   = 'span'
parent_class = None
child_tag    = 'span'
fields       = [
    ('brand_name',          'databind', 'text: name'),
    ('registration_number', 'databind', 'text: regNumber'),
    # ('column_name', 'attr_name', 'attr_value'),
]
output_path  = 'active_ingredients.csv'

print(f"Config ready. Target: {url}")
print(f"Waiting for: '{wait_for}' (timeout: {timeout}ms)")

In [ ]:
# %%
# SECTION 3: fetch_page_playwright()
# ------------------------------------
# Launches a Chromium browser via Playwright, navigates to the configured URL,
# and waits for the 'wait_for' CSS selector to appear before capturing the HTML.
# Playwright's wait_for_selector is more precise than Selenium's — it waits for
# the element to be both present AND visible in the DOM.
# If the selector times out, returns whatever HTML loaded so you can inspect it.
# Result is stored in 'html' for the parse step.
#
# TIP: Set headless=False in Section 2 to watch the browser navigate and debug
# issues like login walls, CAPTCHA, or missing selectors in real time.

def fetch_page_playwright(url, wait_for, timeout, headless):
    with sync_playwright() as p:
        browser = p.chromium.launch(headless=headless)
        page = browser.new_page(user_agent='Mozilla/5.0')
        try:
            page.goto(url, wait_until='domcontentloaded')
            page.wait_for_selector(wait_for, timeout=timeout)
            print(f"Selector '{wait_for}' found — page fully loaded.")
            html = page.content()
        except PlaywrightTimeoutError:
            print(f"Timeout: '{wait_for}' did not appear within {timeout}ms.")
            print("Returning partial HTML. Try increasing timeout or changing wait_for.")
            html = page.content()
        finally:
            browser.close()
    return html

print(f"Fetching: {url}")
html = fetch_page_playwright(url, wait_for, timeout, headless)
print(f"HTML captured. ({len(html)} chars)")

In [ ]:
# %%
# SECTION 4: Parse rendered HTML
# -------------------------------
# Parses the JS-rendered HTML string into a BeautifulSoup object.
# Because Playwright waited for the target selector, this HTML should contain
# the fully rendered content including any data injected by JavaScript.
# Check the preview — you should see actual record data if the page loaded correctly.

soup = BeautifulSoup(html, 'html.parser')
print("HTML parsed.")
print("Preview (first 500 chars of text):")
print(soup.get_text()[:500])

In [ ]:
# %%
# SECTION 5: Inspect parent elements
# ------------------------------------
# Finds parent elements matching your config and prints the first few as raw HTML.
# Use this to verify you're targeting the right container element and understand
# the structure before defining your fields. Adjust parent_tag / parent_class
# in Section 2 and re-run from Section 3 if results look unexpected.

kwargs = {'class_': parent_class} if parent_class else {}
parents = soup.find_all(parent_tag, **kwargs)

print(f"Found {len(parents)} '{parent_tag}' elements.")
print("\nFirst 3 (raw HTML):")
for p in parents[:3]:
    print(p)
    print()

In [ ]:
# %%
# SECTION 6: scrape_records()
# ----------------------------
# Iterates over parent elements and extracts child elements by matching
# attribute name/value pairs from the 'fields' config. Returns one dict
# per parent, with field names as keys and extracted text as values.
# Skips parents where all fields come back empty (e.g. navigation spans).
# Preview the first 5 records before writing to confirm data looks right.

def scrape_records(soup, parent_tag, parent_class, child_tag, fields):
    kwargs = {'class_': parent_class} if parent_class else {}
    parents = soup.find_all(parent_tag, **kwargs)
    records = []
    for parent in parents:
        record = {}
        for col_name, attr_name, attr_value in fields:
            child = parent.find(child_tag, attrs={attr_name: attr_value})
            record[col_name] = child.get_text(strip=True) if child else ''
        if any(record.values()):
            records.append(record)
    return records

records = scrape_records(soup, parent_tag, parent_class, child_tag, fields)
print(f"Extracted {len(records)} records.")
print("\nFirst 5:")
for r in records[:5]:
    print(r)

In [ ]:
# %%
# SECTION 7: write_csv()
# -----------------------
# Writes the extracted records to a multi-column CSV at output_path.
# Column headers come from the field names defined in Section 2.
# Run this last, after the Section 6 preview looks correct.

def write_csv(records, fieldnames, output_path):
    with open(output_path, 'w', newline='', encoding='utf-8') as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(records)
    print(f"Wrote {len(records)} records to '{output_path}'")

if records:
    write_csv(records, [f[0] for f in fields], output_path)
else:
    print("No records to write. Check your field definitions or wait_for selector.")